## Retrieval-Augmented Generation (RAG)

Large Language Models (LLMs) generate responses primarily from knowledge acquired during training. Consequently, they do not automatically have access to specialised or private collections, such as a locally stored research corpus. **Retrieval-Augmented Generation (RAG)** addresses this limitation by combining an LLM with an external information retrieval system.[^1]

Instead of relying exclusively on the model's internal knowledge, a RAG system first **retrieves information relevant to the user's query** and then provides this information to the LLM as additional context. This is particularly useful when working with specialised corpora that were not part of the model's training data, or collections that are too large to fit into the model's context window.[^1]

### A simplified RAG pipeline can be represented as:

> **User question → retrieve relevant documents → add documents to the context → LLM → generated answer**

RAG does not normally retrain the language model on the external collection. Instead, the external data are made available to the model **at inference time**.[^1]




In [ ]:
%pip install --upgrade --force-reinstall \
    "pydantic>=2.12,<2.13" \
    langchain \
    langchain-openai \
    langchain-chroma \
    langchain-docling \
    langchain-community \
    langchain-text-splitters \
    chromadb \
    docling \
    beautifulsoup4 \
    "numpy<2" \
    "pandas>=2.2,<3"

  Using cached langchain-1.4.0-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain_openai-1.6.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_chroma-1.1.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached langchain_docling-3.0.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
  Using cached chromadb-1.5.9-cp39-abi3-macosx_11_0_arm64.whl.metadata (5.0 kB)
  Using cached docling-2.127.0-py3-none-any.whl.metadata (11 kB)
  Using cached beautifulsoup4-4.15.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached numpy-1.26.4-cp312-cp312-macosx_11_0_arm64.whl.metadata (61 kB)
  Using cached pandas-2.3.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (91 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached typing_inspection-0.4.4-py3-none-any.whl.metadata (2.6 kB)
  Using cached langchain_core-1.6

### Installations (skip if not neccessary)

In [3]:
import sys
!{sys.executable} -m pip install -U langchain-community

In [1]:
import importlib.metadata as md

for package in [
    "docling",
    "langchain-docling",
    "pydantic",
    "langchain",
    "numpy",
]:
    print(package, md.version(package))

from langchain_docling import DoclingLoader

print("Docling import successful")

docling 2.127.0
langchain-docling 3.0.0
pydantic 2.8.2
langchain 1.4.0
numpy 1.26.4


/opt/anaconda3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_impl" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_spec" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_name" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_version" has conflict with protected namespace "model_".

You may be able to

AttributeError: force_full_page_ocr

In [2]:
import sys
import importlib.metadata as md

print("Python:", sys.executable)

for package in ["docling", "langchain-docling", "pydantic", "pydantic-core"]:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

Python: /opt/anaconda3/bin/python
docling: 2.127.0
langchain-docling: 3.0.0
pydantic: 2.8.2
pydantic-core: 2.20.1


In [4]:
import sys
import numpy as np

print(sys.executable)
print(np.__version__)
print(np.__file__)

/opt/anaconda3/bin/python
1.26.4
/opt/anaconda3/lib/python3.12/site-packages/numpy/__init__.py


In [3]:
import sys

!{sys.executable} -m pip install \
    --upgrade \
    --force-reinstall \
    --no-cache-dir \
    "pydantic==2.13.5"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 10.9 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.16.0
    Uninstalling typing_extensions-4.16.0:
      Successfully uninstalled typing_extensions-4.16.0
  Attempting uninstall: annotated-types
    Found existing installation: annotated-types 0.6.0
    Uninstalling annotated-types-0.6.0:
      Successfully uninstalled annotated-types-0.6.0
  Attempting uninstall: typing-inspection
    Found existing installation: typing-inspection 0.4.4
    Uninstalling typing-inspection-0.4.4:
      Successfully uninstalled typing-inspection-0.4.4
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.20.1
    Uninstalling pydantic_core-2.20.1:
      Successfully uninstalled pydantic_core-2.20.1
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.8.2
    Uninstalling pydantic-2.8.2:
      Successfully uninstalled pydantic-2.8

In [1]:
import sys
import pydantic
import importlib.metadata as md

print("Python:", sys.executable)
print("Pydantic:", pydantic.__version__)
print("Pydantic location:", pydantic.__file__)
print("Docling:", md.version("docling"))
print("LangChain Docling:", md.version("langchain-docling"))

from langchain_docling import DoclingLoader

print("Docling import successful")

Python: /opt/anaconda3/bin/python
Pydantic: 2.13.5
Pydantic location: /opt/anaconda3/lib/python3.12/site-packages/pydantic/__init__.py
Docling: 2.127.0
LangChain Docling: 3.0.0
Docling import successful


In [2]:
import sys
!{sys.executable} -m pip check

thinc 8.3.6 has requirement numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4.
opencv-python 5.0.0.93 has requirement numpy>=2; python_version >= "3.9", but you have numpy 1.26.4.
streamlit 1.37.1 has requirement protobuf<6,>=3.20, but you have protobuf 7.35.1.


### Main imports

In [3]:
import os
import warnings
import logging

import bs4

from langchain.agents import AgentState, create_agent
from langchain.messages import MessageLikeRepresentation
from langchain.tools import tool

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_docling import DoclingLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

### Environment setup

For this step you will need to: 
- get a Langchain API Key (https://docs.langchain.com/oss/python/deepagents/rag)
- be added DHInfra project by Florian and get DHInfa API kez

In [4]:
from openai import OpenAI

In [84]:

os.environ["LANGCHAIN_API_KEY"] = "" # insert your own Langchain key
os.environ["LANGCHAIN_TRACING_V2"] = "false"  # <-- FIX 1: Disabled to prevent 403 error
os.environ["LANGCHAIN_PROJECT"] = "DHInfra-Tracing-Demo"
os.environ["LANGSMITH_DISABLE_RUN_COMPRESSION"] = "true"
os.environ["USER_AGENT"] = "my_agent"
os.environ["DHINFRA_API_KEY"] = "" # insert DHInfra key

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [6]:
# <-- FIX 2: Custom class to prevent the 422 "null content" error
class SanitizedChatOpenAI(ChatOpenAI):
    def _get_request_payload(self, input_, *args, **kwargs):
        payload = super()._get_request_payload(input_, *args, **kwargs)
        if "messages" in payload:
            for msg in payload["messages"]:
                if msg.get("content") is None:
                    msg["content"] = ""
        return payload

# Initialize chat model using the sanitized class
model = SanitizedChatOpenAI(
    model="qwen3.5-397b",
    openai_api_key=os.environ["DHINFRA_API_KEY"],
    openai_api_base="https://api.dhinfra.uni-graz.at/v1",
    model_kwargs={"parallel_tool_calls": False}
)

# Initialize embedding model
embeddings = OpenAIEmbeddings(
    model="qwen3-embedding-8b",
    openai_api_key=os.environ["DHINFRA_API_KEY"],
    openai_api_base="https://api.dhinfra.uni-graz.at/v1"
)

# Initialize Chroma vector store
vector_store = Chroma(
    collection_name="rag_collection",
    embedding_function=embeddings
)

print("Chat model (Qwen), embedding model (Qwen3-Embedding-8B), and Chroma vector store setup done")

Chat model (Qwen), embedding model (Qwen3-Embedding-8B), and Chroma vector store setup done


## Adapting the code for our own data

If it is a dataframe:

In [7]:
import warnings
import logging
import os
import pandas as pd

warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores.utils import filter_complex_metadata


In [8]:
import pandas as pd
from langchain_core.documents import Document

### Load local CSV

In [9]:
df = pd.read_csv("MigraAnno.csv")
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

In [10]:
print(df.columns)


Index(['id', 'text', 'Topic', 'Name-original', 'newspaper_title', 'date',
       'preceding_document', 'following_document', 'Relevancy_proba',
       'sentiment', 'year', 'Category'],
      dtype='object')


### Convert CSV rows into LangChain Documents

In [19]:
def clean_text(value) -> str:
    return "" if pd.isna(value) else str(value).strip()

def optional_int(value):
    """Convert a value to int, returning None if conversion is impossible."""
    if pd.isna(value):
        return None

    try:
        return int(float(value))
    except (TypeError, ValueError):
        return None


def optional_float(value):
    """Convert a value to float, returning None if conversion is impossible."""
    if pd.isna(value):
        return None

    try:
        return float(value)
    except (TypeError, ValueError):
        return None


In [21]:
from langchain_core.documents import Document

docs = []

for row_index, row in df.iterrows():
    chunk_id = clean_text(row.get("id")) or f"row-{row_index}"
    year = optional_int(row.get("year"))
    relevancy = optional_float(row.get("Relevancy_proba"))

    metadata = {
        "chunk_id": chunk_id,
        "topic": clean_text(row.get("Topic")),
        "name_original": clean_text(row.get("Name-original")),
        "newspaper_title": clean_text(row.get("newspaper_title")),
        "date": clean_text(row.get("date")),
        "preceding_document": clean_text(row.get("preceding_document")),
        "following_document": clean_text(row.get("following_document")),
        "sentiment": clean_text(row.get("sentiment")).lower(),
        "category": clean_text(row.get("Category")),
    }

    if year is not None:
        metadata["year"] = year

    if relevancy is not None:
        metadata["relevancy_proba"] = relevancy

    page_content = f"""
Topic: {metadata["topic"]}
Original label: {metadata["name_original"]}
Category: {metadata["category"]}
Sentiment: {metadata["sentiment"]}

{clean_text(row.get("text"))}
""".strip()

    docs.append(
        Document(
            page_content=page_content,
            metadata=metadata,
        )
    )

print(f"Created {len(docs)} documents")
print(docs[0])

Created 96871 documents
page_content='Topic: 0
Original label: 0_hungarn_armee_feind_böhmen
Category: CTX
Sentiment: neutral

DEn  dito hat man dass die Land Militz welche hin und wieder in Oesterreich auffgebotten worden theils zu Stockerau theils anderwerts ihren vorgesetzten Officiers diser Tagen übergeben worden umb sie in denen Waffen zu  exerciren.' metadata={'chunk_id': '1.0', 'topic': '0', 'name_original': '0_hungarn_armee_feind_böhmen', 'newspaper_title': 'Diarium', 'date': '1703-08-13', 'preceding_document': 'dess Herrn Bischoffs zu Eutin Reichs  Contingent, wie auch dess Herrn Obristen von Osten Hollsteinische Dragoner Regiment so alle wol  montirt und brave Leüthe sind den  hujus auss den Eutinischen auffgebrochen und haben zu Schwartau ihr Rendevous gehalten daselbst sie ins gesambt von Jhrer Durchl. Wie verlautet seynd  Käyserl.', 'following_document': 'Durchl. sehr bedauren; andern Tags als den  dito seyn  Blessirte zu besagtem München ankommen welche gar miserabel gesta

In [22]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="migraanno_newspapers_v2",
    embedding_function=embeddings,
    persist_directory="./chroma_migraanno",
)

A subset of the corpus

In [23]:
batch_size = 100

for start in range(0, len(docs), batch_size):
    batch = docs[start:start + batch_size]

    batch_ids = [
        f"migraanno-{index}"
        for index in range(start, start + len(batch))
    ]

    vector_store.add_documents(
        documents=batch,
        ids=batch_ids,
    )

    end = min(start + batch_size, len(docs))
    print(f"Indexed {end}/{len(docs)} documents")

Indexed 100/96871 documents
Indexed 200/96871 documents
Indexed 300/96871 documents
Indexed 400/96871 documents
Indexed 500/96871 documents
Indexed 600/96871 documents
Indexed 700/96871 documents
Indexed 800/96871 documents
Indexed 900/96871 documents
Indexed 1000/96871 documents
Indexed 1100/96871 documents
Indexed 1200/96871 documents
Indexed 1300/96871 documents
Indexed 1400/96871 documents
Indexed 1500/96871 documents
Indexed 1600/96871 documents
Indexed 1700/96871 documents
Indexed 1800/96871 documents
Indexed 1900/96871 documents
Indexed 2000/96871 documents
Indexed 2100/96871 documents
Indexed 2200/96871 documents
Indexed 2300/96871 documents
Indexed 2400/96871 documents
Indexed 2500/96871 documents
Indexed 2600/96871 documents
Indexed 2700/96871 documents
Indexed 2800/96871 documents
Indexed 2900/96871 documents
Indexed 3000/96871 documents
Indexed 3100/96871 documents
Indexed 3200/96871 documents
Indexed 3300/96871 documents
Indexed 3400/96871 documents
Indexed 3500/96871 docu

In [25]:
print("Documents in Chroma:", vector_store._collection.count())

Documents in Chroma: 96871


In [33]:
records_1889 = vector_store.get(
    where={"year": 1889},
    include=["metadatas"],
)

print("Documents from 1889:", len(records_1889["ids"]))
print(records_1889["metadatas"][:3])

Documents from 1889: 748
[{'preceding_document': 'Und so wird es bleiben, so lange die Bäcker und so viele andere Arbeiter zu Hause, im eigenen Lande, sich Zustände gefallen lassen, gegen welche fortgeschrittene Arbeiter nicht nur in Worten, sondern mit Thaten protestiren würden. Ueberflüssige Kopfschmerzen macht unseren offiziösen Blättern das Zwanzigjährige Gründungsfest des Arbeiter=Bildungsvereins „Vorwärts“ in Preßburg.', 'topic': '0', 'following_document': 'Selbst diesem harmlosen Blättchen war die Geschichte zu toll, wie die folgenden Notizen zeigen, die zwischen dem 12. und 15. September nacheinander erschienen. Näherer! Die offiziösen Wiener Journale „Presse“ und „Fremdenblatt“ enthalten in ihren gestrigen Frühnummern ein Telegramm aus Preßburg folgenden Inhaltes: „Die in Wien verbotene Lassalle=Feier soll durch Wiener Arbeiter hier im nächsten Monat begangen werden.', 'newspaper_title': 'aze', 'year': 1889, 'category': 'MIN', 'chunk_id': '189.0', 'relevancy_proba': 0.9999347,

## Loading the embeddings (so no need for re-indexing)

In [88]:
import os
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# recreate the same embedding model used during indexing.
embeddings = OpenAIEmbeddings(
    model="qwen3-embedding-8b",
    openai_api_key=os.environ["DHINFRA_API_KEY"],
    openai_api_base="https://api.dhinfra.uni-graz.at/v1",
)

# load the existing Chroma collection from disk.
vector_store = Chroma(
    collection_name="migraanno_newspapers_v2",
    embedding_function=embeddings,
    persist_directory="./chroma_migraanno",
)

# verify that the saved documents were loaded.
print("Loaded documents:", vector_store._collection.count())

Loaded documents: 96871


## Search newspapers function

In [53]:
def search_newspapers(
    query: str,
    year: int | None = None,
    sentiment: str | None = None,
    category: str | None = None,
    k: int = 10,
):
    conditions = []

    if year is not None:
        conditions.append({
            "year": {"$eq": int(year)}
        })

    if sentiment:
        conditions.append({
            "sentiment": {"$eq": sentiment.strip().lower()}
        })

    if category:
        conditions.append({
            "category": {"$eq": category.strip()}
        })

    if not conditions:
        metadata_filter = None
    elif len(conditions) == 1:
        metadata_filter = conditions[0]
    else:
        metadata_filter = {"$and": conditions}

    return vector_store.similarity_search(
        query=query,
        k=k,
        filter=metadata_filter,
    )

## Newspaper Search Function

The `search_newspapers()` function performs **semantic retrieval** over the newspaper texts stored in Chroma while optionally applying exact metadata filters.

```python
def search_newspapers(
    query: str,
    year: int | None = None,
    sentiment: str | None = None,
    category: str | None = None,
    k: int = 10,
):
    ...
```

### Parameters

| Parameter | Type | Default | Description |
|---|---:|---:|---|
| `query` | `str` | Required | Semantic search query. It may contain English, German, or multilingual search terms. |
| `year` | `int \| None` | `None` | Restricts retrieval to documents from one exact year. |
| `sentiment` | `str \| None` | `None` | Restricts retrieval to an exact sentiment label, such as `"negative"`, `"neutral"`, or `"positive"`. The value is converted to lowercase. |
| `category` | `str \| None` | `None` | Restricts retrieval to an exact category value, such as `"MIG"` or `"CTX"`. |
| `k` | `int` | `10` | Maximum number of documents returned. |

### Retrieval procedure

The function combines two retrieval mechanisms:

1. **Semantic similarity search**  
   Chroma compares the meaning of `query` with the embedded newspaper texts.

2. **Metadata filtering**  
   The optional `year`, `sentiment`, and `category` parameters restrict the documents considered during semantic search.

When several filters are supplied, they are combined with Chroma's `$and` operator. Therefore, a document must satisfy all specified conditions.

### Return value

The function returns a list of LangChain `Document` objects:

```python
list[Document]
```

Each result contains:

- `result.page_content`: the retrieved newspaper text;
- `result.metadata`: associated metadata such as date, year, newspaper, category, sentiment, and neighbouring text.

### Examples

#### Semantic search without metadata filters

```python
results = search_newspapers(
    query="minority populations and ethnic groups",
    k=10,
)
```

#### Search within one year

```python
results = search_newspapers(
    query="minority populations and ethnic groups",
    year=1889,
    k=10,
)
```

#### Search by year and sentiment

```python
results = search_newspapers(
    query="hostility towards ethnic minorities",
    year=1889,
    sentiment="negative",
    k=20,
)
```

#### Search by year, sentiment, and category

```python
results = search_newspapers(
    query="Croats and Serbs, Kroaten und Serben, Nationalitätenfrage",
    year=1889,
    sentiment="negative",
    category="MIG",
    k=20,
)
```

### Important notes

- Metadata filters use **exact matching**.
- `"MIG"` and `"mig"` are different category values unless categories were normalized before indexing.
- The year must be stored in Chroma as an integer for `year=1889` to match.
- `k` specifies the maximum number of returned results, not necessarily the total number of matching documents.
- English queries can retrieve German texts when a multilingual embedding model is used.
- Including German terms and historical spelling variants can improve retrieval from historical newspapers.

## Query 1 

In [44]:
results = vector_store.similarity_search(
    query=(
        "minorities, ethnic minorities, linguistic minorities, "
        "religious minorities, minority populations"
    ),
    k=10,
    filter={"year": 1889},
)

In [45]:
for number, result in enumerate(results, start=1):
    print("=" * 80)
    print(f"Result {number}")
    print("ID:", result.metadata.get("chunk_id"))
    print("Date:", result.metadata.get("date"))
    print("Newspaper:", result.metadata.get("newspaper_title"))
    print("Topic:", result.metadata.get("topic"))
    print("Category:", result.metadata.get("category"))
    print("Sentiment:", result.metadata.get("sentiment"))
    print()
    print(result.page_content[:800])

Result 1
ID: 220165.0
Date: 1889-11-01
Newspaper: nfp
Topic: 7
Category: CTX
Sentiment: positive

Topic: 7
Original label: 7_wahlen_wähler_abstimmung_wahl
Category: CTX
Sentiment: positive

Diesen Trost gewährt die diesmalige Budget=Debatte des Reichstages, und dadurch empfängt sie ihre Bedeutung. Es ist von allen Parteien wie vom Regierungstische hinausgesprochen worden zu dem Volke; gute Vorsätze und verschämte Bekenntnisse, scharfe Anschuldigungen und beschwichtigende Friedensbethenerungen, kühne Beweisführungen und mühsame Widerlegungen — die bunte Fülle wird sich in dem Intellecte des Wählers von selbst sichten, und erst am Wahltage wird um das Echo der Reden vernehmen, welche seit vorgestern auf den deutschen Reichstag die allgemeine Aufmerksamkeit gelenkt haben.)
Result 2
ID: 118331.0
Date: 1889-12-01
Newspaper: vtl
Topic: 13
Category: MIN
Sentiment: negative

Topic: 13
Original label: 13_juden_deutschen_jüdischen_jüdische
Category: MIN
Sentiment: negative

Das Factum, daß ein K

## Query 2

In [61]:
results = search_newspapers(
    query="refugees",
    year=1905,
    sentiment ="negative",
    category = "MIG",
    k=20,
)

In [65]:
for number, result in enumerate(results, start=1):
    metadata = result.metadata

    print("=" * 100)
    print(f"Result {number}")
    print("ID:", metadata.get("chunk_id"))
    print("Date:", metadata.get("date"))
    print("Year:", metadata.get("year"))
    print("Newspaper:", metadata.get("newspaper_title"))
    print("Topic:", metadata.get("topic"))
    print("Category:", metadata.get("category"))
    print("Sentiment:", metadata.get("sentiment"))

    print("\n--- PRECEDING TEXT ---")
    print(metadata.get("preceding_document") or "[No preceding text]")

    print("\n--- RETRIEVED TEXT ---")
    print(result.page_content or "[No retrieved text]")

    print("\n--- FOLLOWING TEXT ---")
    print(metadata.get("following_document") or "[No following text]")

    print()

Result 1
ID: 177820.0
Date: 1884-08-01
Year: 1884
Newspaper: nfp
Topic: 3
Category: MIN
Sentiment: negative

--- PRECEDING TEXT ---
DaS ofsicielle Organ fährt dann, nachdem esden Wünschen der Italiener Tirols Ausdruck gegeben, in folgender Weise fort: „Ohne Bedenken müssen wir behauplen, daß alleSloveneu, auch die radicalsten, Gott auf den Knien danken wurden, wenn man der slovenischen Sache einmal so viele Concessionen vergönnen möchte, wie .solche so reichlich die Trientincrfett Langem in Frieden genießen.

--- RETRIEVED TEXT ---
Topic: 3
Original label: 3_czechen_polen_deutschen_czechischen
Category: MIN
Sentiment: negative

Die Czechen bekämpfen diese Forderungen der Wälschtiroler, zugleich aber auch, obwol indirect, das heiße Sehnen der Slovenen nach Vereinigung derselben zu einem staatsrechtlichen Ganzen

--- FOLLOWING TEXT ---
DaS vereinigteSlovsnien würde auf unsere nationale Entwicklung einen mächtigen und wohlthätigen Einfluß üben; mit der Förderung _ derslovenischen Sprache 

## Query 3

In [66]:
results = search_newspapers(
    query="education and minorities",
    sentiment= "negative",
    year=1884,
    k=10,
)

In [67]:
for number, result in enumerate(results, start=1):
    metadata = result.metadata

    print("=" * 100)
    print(f"Result {number}")
    print("ID:", metadata.get("chunk_id"))
    print("Date:", metadata.get("date"))
    print("Year:", metadata.get("year"))
    print("Newspaper:", metadata.get("newspaper_title"))
    print("Topic:", metadata.get("topic"))
    print("Category:", metadata.get("category"))
    print("Sentiment:", metadata.get("sentiment"))

    print("\n--- PRECEDING TEXT ---")
    print(metadata.get("preceding_document") or "[No preceding text]")

    print("\n--- RETRIEVED TEXT ---")
    print(result.page_content or "[No retrieved text]")

    print("\n--- FOLLOWING TEXT ---")
    print(metadata.get("following_document") or "[No following text]")

    print()

Result 1
ID: 177820.0
Date: 1884-08-01
Year: 1884
Newspaper: nfp
Topic: 3
Category: MIN
Sentiment: negative

--- PRECEDING TEXT ---
DaS ofsicielle Organ fährt dann, nachdem esden Wünschen der Italiener Tirols Ausdruck gegeben, in folgender Weise fort: „Ohne Bedenken müssen wir behauplen, daß alleSloveneu, auch die radicalsten, Gott auf den Knien danken wurden, wenn man der slovenischen Sache einmal so viele Concessionen vergönnen möchte, wie .solche so reichlich die Trientincrfett Langem in Frieden genießen.

--- RETRIEVED TEXT ---
Topic: 3
Original label: 3_czechen_polen_deutschen_czechischen
Category: MIN
Sentiment: negative

Die Czechen bekämpfen diese Forderungen der Wälschtiroler, zugleich aber auch, obwol indirect, das heiße Sehnen der Slovenen nach Vereinigung derselben zu einem staatsrechtlichen Ganzen

--- FOLLOWING TEXT ---
DaS vereinigteSlovsnien würde auf unsere nationale Entwicklung einen mächtigen und wohlthätigen Einfluß üben; mit der Förderung _ derslovenischen Sprache 

In [74]:
results = search_newspapers(
    query=(
        "Croats and Serbs; Croatian and Serbian populations, "
        "Croatien, Kroatien, Serbien, Serben, Kroaten, "
        "Serbo-Kroaten, Serbokroaten, Südslawen"
    ),
    year=1889,
    k=20,
)

In [75]:
for number, result in enumerate(results, start=1):
    metadata = result.metadata

    print("=" * 100)
    print(f"Result {number}")
    print("ID:", metadata.get("chunk_id"))
    print("Date:", metadata.get("date"))
    print("Year:", metadata.get("year"))
    print("Newspaper:", metadata.get("newspaper_title"))
    print("Topic:", metadata.get("topic"))
    print("Category:", metadata.get("category"))
    print("Sentiment:", metadata.get("sentiment"))

    print("\n--- PRECEDING TEXT ---")
    print(metadata.get("preceding_document") or "[No preceding text]")

    print("\n--- RETRIEVED TEXT ---")
    print(result.page_content or "[No retrieved text]")

    print("\n--- FOLLOWING TEXT ---")
    print(metadata.get("following_document") or "[No following text]")

    print()

Result 1
ID: 214937.0
Date: 1889-03-01
Year: 1889
Newspaper: nfp
Topic: 225
Category: CTX
Sentiment: positive

--- PRECEDING TEXT ---
Ichhoffe, eS wird nicht dahin kommen.

--- RETRIEVED TEXT ---
Topic: 225
Original label: 225_monarchie_dynastie_monarchisten_kaiserreich
Category: CTX
Sentiment: positive

Wir lieben unser Oesterreich treu und beharrlich, wir lieben Oesterreich, nicht das des Papstes, sondern das des Kaisers von Oesterreich, nicht die Raritätenkammer für czechtsches Staatsrecht und für hageilonische Ideen, sondern das Oesterreich, wie es auf gesetzlicher Grundlage gewachsen ist, sich aufgebaut hat und wie s weiter gedeihen nöge!

--- FOLLOWING TEXT ---
Weil wir Oesterreich lieben, darum sindwir entschieden und beharrlich gegen diese Regierung.

Result 2
ID: 117391.0
Date: 1889-08-01
Year: 1889
Newspaper: vtl
Topic: 0
Category: MIN
Sentiment: negative

--- PRECEDING TEXT ---
In jenem Vertrage ist nämlich festgesetzt, daß Derjenige, der das Gastrecht in einem der beiden St

In [82]:
def build_multilingual_query(
    english: str,                         # English search 
    german: str,                          # German search 
    historical_terms: list[str] | None = None,  # Optional historical variants
) -> str:

    # C=create the main English and German parts
    parts = [
        f"English concepts: {english.strip()}",
        f"German concepts: {german.strip()}",
    ]

    # add historical terms if they were provided.
    if historical_terms:
        parts.append(
            "Historical terms and spelling variants: "
            + ", ".join(historical_terms)
        )

    return "\n".join(parts)

In [80]:
query = build_multilingual_query(
    english=(
        "Croats and Serbs, ethnic relations, national identity, "
        "political conflict and minority rights"
    ),
    german=(
        "Kroaten und Serben, ethnische Beziehungen, nationale Identität, "
    ),
    historical_terms=[
        "Croaten",
        "Croatien",
        "Serbien",
        "Serben",
        "Südslawen",
    
    ],
)

results = search_newspapers(
    query=query,
    category="MIN",
    k=20,
)

In [81]:
for number, result in enumerate(results, start=1):
    metadata = result.metadata

    print("=" * 100)
    print(f"Result {number}")
    print("ID:", metadata.get("chunk_id"))
    print("Date:", metadata.get("date"))
    print("Year:", metadata.get("year"))
    print("Newspaper:", metadata.get("newspaper_title"))
    print("Topic:", metadata.get("topic"))
    print("Category:", metadata.get("category"))
    print("Sentiment:", metadata.get("sentiment"))

    print("\n--- PRECEDING TEXT ---")
    print(metadata.get("preceding_document") or "[No preceding text]")

    print("\n--- RETRIEVED TEXT ---")
    print(result.page_content or "[No retrieved text]")

    print("\n--- FOLLOWING TEXT ---")
    print(metadata.get("following_document") or "[No following text]")

    print()

Result 1
ID: 107166.0
Date: 1875-09-01
Year: 1875
Newspaper: nfp
Topic: 27
Category: MIN
Sentiment: negative

--- PRECEDING TEXT ---
Auch kann man wohl annehmen, daß sichOesterreich für den Fall, daß es im Interesse der Drei«Kaiser-Politik in die Action treten sollt?, sowol bezüglich derpolitischen Coitsequenzen, als der materiellen Opferder Schadloshaltung seitens seiner Verbündeten vergewissert habe."

--- RETRIEVED TEXT ---
Topic: 27
Original label: 27_bosnien_serben_serbien_montenegro
Category: MIN
Sentiment: negative

— Die Welt würde mit sehr wenig Weisheit regiert werden, wenn die drei Mächte den Fall eines serbischen Excesses nicht vorgesehen hätten, aber die Richtigkeit der obigen Angaben scheint uns denndoch im höchsten Grade zweifelhaft, und die Schlesische Zeitung thut Recht, diese Mittheilung mit einem skeptischen Fragezeichen zu breiten. Wien, .

--- FOLLOWING TEXT ---
(Preßstimmen über den Aufstand.)

Result 2
ID: 174556.0
Date: 1884-02-01
Year: 1884
Newspaper: nfp
Topic